<h1 align="center">Implementing Decision Trees & Random Forests from Scratch</h1>

This Project Majorly consists of three parts:
1. Decision Tree Implementation
2. Random Forest Ensemble Logic
3. Model Performance & Comparative Analysis

**Import libraries**

In [20]:
# ==============================
# TO BE IMPLEMENTED
# ==============================

# import are relvant libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.api.types import is_numeric_dtype

### **Part 1 — Custom Decision Tree**

### Load and Prepare Titanic Dataset

In [21]:
# Load training data
data = pd.read_csv("titanic.csv")

print(data.head())
print(data.info())
print(data.isnull().sum())



   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  
<c

The Titanic dataset contains passenger information such as age, class, gender, fare, and survival status. Some columns contain missing values and categorical data that need to be handled properly.

---

## **Task: Implement Data Preprocessing Functions**

We are required to write Python functions (without using advanced preprocessing libraries like `sklearn.preprocessing`) to perform the following steps manually.

---

### *Step 1: Handle Missing Values*

Write functions to:

1. Replace missing values in the **Age** column with the **median age** of all passengers.  
2. Replace missing values in the **Embarked** column with the **most common embarkation port (mode)**.

---

### *Step 2: Drop Irrelevant Columns*

Remove the following columns that are not directly useful for survival prediction:

1. **Cabin** – too many missing values  
2. **Ticket** – non-numeric and not informative  
3. **Name** – mostly unique values  
4. **PassengerId** – identifier, not a feature

---

### **Step 3: Encode Categorical Variables**

Since machine learning models require numerical data, we need to convert categorical variables into numeric format manually.

#### Encode the `Sex` column:
- `male` → `0`  
- `female` → `1`

#### Encode the `Embarked` column:
- `C` → `0`  
- `Q` → `1`  
- `S` → `2`

This manual encoding ensures that categorical features can be interpreted correctly by machine learning algorithms, which typically work only with numerical inputs.


In [ ]:

# Handle missing values
def handle_missing_values(df):
    """ 
    Handle missing values in Age ( replace missing with median) and Embarked ( replace missing with mode) , 
    """

    if  'Age' in df.columns :
        df['Age'] = df['Age'].fillna(df['Age'].median()) # replace missing 'Age' with median age 
         

    if 'Embarked' in df.columns:
        mode_embarked = df['Embarked'].mode(dropna=True)[0] # replace the missing 'Embarked' with mode 
        df['Embarked'] = df['Embarked'].fillna(mode_embarked)



    return df

# Drop unused columns
def drop_irrelevant_coloumns(df):
    """
    Drop the columns that are not uueful in prediction 
    """
    columns_to_drop = [ 'Cabin', 'Ticket', 'Name', 'PassengerId']
    df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True, errors='ignore')
    return df

    
# Encode categorical features manually
def encode_catagorical(df):
    """
    Drop categorical variables manually
    """
    if 'Sex' in df.columns: 
        df[ 'Sex' ] = df['Sex'].map({ 'male':0, 'female':1})
    
    if 'Embarked' in df.columns:
        df['Embarked'] = df[ 'Embarked'].map({'C' : 0, 'Q' : 1, 'S' : 2})

    return df 
# Clean and encode the dataset
data = handle_missing_values(data)
data = drop_irrelevant_coloumns(data)
data = encode_catagorical(data)

X = data.drop('Survived', axis=1)
y = data['Survived'].values

print(data[['Sex', 'Embarked']].isnull().sum())


Sex         0
Embarked    0
dtype: int64


Split data into train and test (Do NOT change following cell)

In [23]:
X = data.drop('Survived', axis=1)
y = data['Survived'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## **Task: Implement Core Functions for Decision Tree Splitting**

In this section, we implement the key mathematical components that allow a **Decision Tree** to decide where to split data during training. You will write three main functions: `entropy`, `information_gain`, and `best_split`.

---

### **Step 1: Compute Entropy**
- The entropy function measures the impurity or uncertainty in a dataset.  
- Formula:  
  $$
  H(y) = -\sum_i p_i \log_2(p_i)
  $$
  where $p_i$ is the probability of each class label.
- Use NumPy operations to calculate probabilities.
- Add a small constant (e.g., `1e-9`) inside the log to avoid numerical errors.
- Test on simple arrays, e.g., `entropy([0, 0, 1, 1])`.

---

### **Step 2: Compute Information Gain**
- Information Gain (IG) quantifies how much entropy decreases after a split.
- Formula:  
  $$
  IG = H(\text{parent}) - \frac{n_{left}}{n}H(\text{left}) - \frac{n_{right}}{n}H(\text{right})
  $$
- Implement this using the `entropy()` function.
- The higher the IG, the better the feature/threshold for splitting.

---

### **Step 3: Find the Best Split**
- The `best_split()` function finds which feature and threshold produce the **maximum information gain**.
- Loop through all features (or a given subset) and:
  - For **numerical features**:
    - Try splitting at all unique threshold values.
    - Create boolean masks for left (`<= t`) and right (`> t`) splits.
  - For **categorical features**:
    - Split data based on equality (`== val`) vs. inequality (`!= val`).
- Skip invalid splits (e.g., when one side is empty).
- Return:
  - `best_feature`: the most informative feature,
  - `best_threshold`: the split value,
  - `best_gain`: the highest information gain achieved.

---



In [ ]:

import numpy as np
EPS = 1e-9
def entropy(y):
    """
    Compute the entropy of a label array y.
    Formula: H(y) = -Σ p_i * log2(p_i)
    """
    y = np.asarray(y)
    if y.size == 0 :
        return 0
    
    
    unique, counts = np.unique (y, return_counts=True)  # count for each unique label 
    probs = counts / counts.sum()     
    return -np.sum(probs * np.log2(probs + EPS))  # add constant to avoid log (0) 



def information_gain(y, y_left, y_right):
    """
    Compute the information gain of a split.
    IG = H(parent) - (n_left/n)*H(left) - (n_right/n)*H(right)
    """
    y = np.asarray(y) 
    if y.size == 0: 
        return 0
    
    n = len(y)
    n_left = len(y_left)
    n_right = len(y_right)

    if n_left == 0 or n_right == 0:  # zero information gain if child is empty (i.e. invalid split )
        return 0
     
    
    H_parent  = entropy(y)
    H_left = entropy (y_left)
    H_right = entropy (y_right)
    
    return H_parent - (n_left / n) * H_left - (n_right / n) * H_right
    


def best_split(X, y, feature_subset=None):
    """
    Find the best feature and threshold that maximize information gain.

    Parameters:
        X (DataFrame): Feature dataset
        y (Series or array): Target labels
        feature_subset (list): Optional subset of features for random forest

    Returns:
        best_feature (str): Feature name with best split
        best_threshold (float/str): Threshold or category value
        best_gain (float): Information gain of best split
    """
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X)
    
    features =  feature_subset if feature_subset is not None else X.columns # use all coloumns if feature_subset is none 
    best_gain = 0
    best_feature = None
    best_threshold = None
    y = np.asarray (y) 

    for feature in features: 
        col = X[feature]
        if col.isnull().all():
            continue    
        
        if is_numeric_dtype(col): # numeric features 
            unique_vals = np.unique(col.dropna())
            if len(unique_vals) <= 1:
                continue
            thresholds = (unique_vals[:-1] + unique_vals[1:]) / 2.0
            for t in thresholds:
                left_mask = col <= t
                right_mask = col > t    
            
                
            # extract target labels 
                y_left = y[left_mask]
                y_right = y[right_mask]

                if len(y_left) == 0 or len(y_right) == 0:  # skip invalid splits 
                    continue
                gain = information_gain(y, y_left, y_right)
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = t
    # Categorical features
        else:
            for val in col.dropna().unique():
                left_mask = col == val
                right_mask = col != val

                y_left = y[left_mask]
                y_right = y[right_mask]

                if len(y_left) == 0 or len(y_right) == 0:
                    continue

                gain = information_gain(y, y_left, y_right) # information gain (categorical feature)
                if gain > best_gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = val
                
    return best_feature, best_threshold, best_gain

## **Task: Implement the Custom Decision Tree Classifier**

In this task, we implement our own **Decision Tree** algorithm from scratch — without using libraries like `sklearn.tree`.  
This implementation will rely on the previously defined helper functions: `entropy`, `information_gain`, and `best_split`.

---

### **Class Overview**

We create a `DecisionTree` class with the following methods:

---

### **1. `__init__()`**
- Initializes the tree’s hyperparameters:
  - `max_depth`: the maximum allowed depth of the tree.  
  - `min_samples_split`: minimum number of samples required to make a split.  
  - `feature_subsample`: number of randomly selected features (used in Random Forests).  
- The tree itself will be stored as a nested dictionary (`self.tree`).

---

### **2. `fit()`**
- Recursively builds the decision tree using **Information Gain** as the splitting criterion.
- **Base cases**:
  - If there are no samples left (`len(y) == 0`) → return `0`.
  - If all samples belong to the same class → return that class label.
  - If maximum depth is reached or too few samples remain → return the most common class (majority vote).
- **Recursive case**:
  1. Select the best feature and threshold using `best_split()`.
  2. Split the dataset into left and right subsets based on that threshold.
  3. Recursively call `fit()` on each subset to build subtrees.
- The final tree should be stored in `self.tree` as nested dictionaries with keys:
..., 'threshold': ..., 'gain': ..., 'left': ..., 'right': ...}


---

### **3. `predict_one()`**
- Predict the class for a **single data point**.
- Traverse the decision tree recursively:
- For numeric thresholds: use `<=` for the left branch, `>` for the right.
- For categorical values: use `==` for the left branch, `!=` for the right.
- Stop recursion when a leaf node (class label) is reached.

---

### **4. `predict()`**
- Predict the class for **an entire dataset** (`X`).
- Apply `predict_one()` to each row using a loop or list comprehension.
- Return predictions as a NumPy array.

---


In [ ]:

import numpy as np
from collections import Counter

class DecisionTree:
    def __init__(self, max_depth=4, min_samples_split=2, feature_subsample=None):
        """
        Initialize the Decision Tree parameters.
        """
        self.max_depth = max_depth 
        self.min_samples_split = min_samples_split
        self.feature_subsample = feature_subsample
        self.tree = None 
        
    def fit(self, X, y, depth=0):
        """
        Recursively build the decision tree using information gain.
        """
        y = np.array(y) # cnvert to NumPy arrays for consistancy 
        
        if len(y) == 0: # no sample case 
            return 0
        
        if len(np.unique(y)) ==1: # all labels same- return that class 
            return y[0]

        if depth >= self.max_depth or len(y) < self.min_samples_split:  #max depth reached or no enough sample to slpit
            return Counter (y). most_common(1)[0][0] # majority vote 
        
        # Select subset of featurs ( for random forest support)
        if self.feature_subsample is not None:
            features = np.random.choice(X.columns,self.feature_subsample, replace = False)
        else:
            features = X.columns 
        
        # find best feature and threshold using helper 
        best_feature, best_threshold, best_gain = best_split(X, y, features)
   
        if best_gain <= 0 or best_feature is None:
            return Counter(y).most_common(1)[0][0]
        
        col = X[best_feature]
        
        
        # split the data 
        if is_numeric_dtype(col): # Numeric split
            left_mask = col <= best_threshold
            right_mask = col > best_threshold
        else:  # Categorical split
            left_mask = col == best_threshold
            right_mask = col != best_threshold
        X_left, y_left = X[left_mask ], y[left_mask ]
        X_right, y_right = X[right_mask], y[right_mask]
        
        
        
        
        # build left and right branches recursively 
        left_subtree = self.fit(X_left, y_left, depth+1)
        right_subtree = self.fit(X_right, y_right, depth+1)
        
        # current node 
        node = {
            'feature': best_feature,
            'threshold': best_threshold,
            'gain': best_gain,
            'left' : left_subtree, 
            'right': right_subtree
    
        }            
   
        # save the full tree if root call 
        if depth == 0:
            self.tree = node 
            
        return node 
    
   
    def predict_one(self, x, node=None):
        """
        Predict class label for a single sample.
        """
        if node is None:
            node = self.tree
            
        #leaf node - class label 
        if not isinstance(node, dict):
            return node 
        feature = node['feature']
        threshold = node ['threshold']
        
        # numeric feature split 
        if isinstance(threshold, (int, float, np.number)):
            if x[feature] <= threshold:
                return self.predict_one(x, node["left"])
            else:
                return self.predict_one(x, node["right"])
        else:
            if x[feature] == threshold:
                return self.predict_one(x, node["left"])
            else:
                return self.predict_one(x, node["right"])
        

    def predict(self, X):
        """
        Predict class labels for all rows in dataset X.
        """
        return np.array([self.predict_one(row) for _, row in X.iterrows()])
        

### **Train and Evaluate the Custom Decision Tree**
 
It will be used to train, test, and evaluate your implemented `DecisionTree` class automatically.


In [34]:
tree = DecisionTree(max_depth=4)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

acc_tree = np.mean(y_pred_tree == y_test)
print(f"Custom Decision Tree Accuracy: {acc_tree:.4f}")

compare_tree = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred_tree})
compare_tree.head(10)


Custom Decision Tree Accuracy: 0.7989


,Actual,Predicted
0,1,0
1,0,0
2,0,0
3,1,1
4,1,1
5,1,1
6,1,1
7,0,0
8,1,1
9,1,1


## **Part 2 — Custom Random Forest**

In this section, we implement your own **Random Forest Classifier** from scratch — without using libraries like `sklearn.ensemble`.

A Random Forest combines multiple Decision Trees to form an ensemble model that improves prediction accuracy and reduces overfitting.

---

### **Instructions**

1. **Initialize Parameters (`__init__`)**
   - `n_estimators`: Number of decision trees to train.
   - `max_depth`: Maximum depth of each decision tree.
   - `min_samples_split`: Minimum number of samples required to split a node.
   - `feature_subsample_ratio`: Fraction of features to randomly select for each tree (e.g., 0.7 = 70% of features per tree).

   Store all trees in a list called `self.trees`.

---

2. **Train the Model (`fit`)**
   - For each tree:
     - Create a **bootstrap sample** by randomly selecting rows from the dataset **with replacement**.
     - Randomly choose a subset of features using the ratio `feature_subsample_ratio`.
     - Train a new instance of your `DecisionTree` class on the sampled data.
   - Append each trained tree to the list `self.trees`.

---

3. **Make Predictions (`predict`)**
   - Each trained decision tree makes predictions for all samples in `X`.
   - Combine predictions from all trees using **majority voting** (most common class).
   - Return the final predicted labels as a NumPy array.

---

### **Goal**
By completing this part, we understand how ensemble learning improves model robustness through randomness and aggregation.


Implement Custom Random Forest Class

In [ ]:

import numpy as np
from collections import Counter

class RandomForest:
    def __init__(self, n_estimators=10, max_depth=5, min_samples_split=2, feature_subsample_ratio=0.7):
        """
        Initialize the Random Forest parameters.
        """
        self.n_estimators = n_estimators 
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.feature_subsample_ratio = feature_subsample_ratio
        self.trees = [] # list to store all trained trees 
    
    
    
    def fit(self, X, y):
        """
        Train multiple Decision Trees using bootstrapped samples and random feature subsets.
        """
        n_samples, n_features = X.shape
        feature_subsample_size = int(np.ceil(self.feature_subsample_ratio * n_features))
        
        for _ in range(self.n_estimators):
            
            # bootstrap sample (with row replacment)
            sample_indices = np.random.choice(n_samples, size=n_samples, replace=True)
            X_sample = X.iloc[sample_indices]
            y_sample = y[sample_indices]
            
            # random subset of features 
            feature_subset = np.random.choice(X.columns, feature_subsample_size, replace = False)
            
            # train decision tree on subset 
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split, 
                feature_subsample=len(feature_subset)    
            )
            tree.fit(X_sample[feature_subset], y_sample)
            self.trees.append((tree, feature_subset))
            

    def predict(self, X):
        """
        Predict class labels for all samples using majority voting from all trained trees.
        """
        tree_predictions = [] 

        # collect predictions from all trees
        for tree, feature_subset in self.trees:
            preds = tree.predict(X[feature_subset])
            tree_predictions.append(preds)
        
        
        
        tree_predictions = np.array(tree_predictions) # convert to numpy array 
        
        # majority voting
        final_preds = []
        for i in range (tree_predictions.shape[1]):
            votes = tree_predictions[:, i]
            majority_vote = Counter(votes).most_common(1)[0][0]
            final_preds.append(majority_vote)
        return np.array(final_preds)    


### **Train and Evaluate the Custom Random Forest**



In [56]:
forest = RandomForest(n_estimators=10, max_depth=5)
forest.fit(X_train, y_train)
y_pred_forest = forest.predict(X_test)

acc_forest = np.mean(y_pred_forest == y_test)
print(f"Custom Random Forest Accuracy: {acc_forest:.4f}")

compare_forest = pd.DataFrame({
    'Actual': y_test,
    'Tree_Pred': y_pred_tree,
    'Forest_Pred': y_pred_forest
})
compare_forest.head(7)


Custom Random Forest Accuracy: 0.8212


,Actual,Tree_Pred,Forest_Pred
0,1,0,0
1,0,0,0
2,0,0,0
3,1,1,1
4,1,1,1
5,1,1,1
6,1,1,1


### **Comparison**
 
Compares the performance of your **Custom Decision Tree** and **Custom Random Forest** implementations by printing their respective accuracies.


In [29]:
print(f"Decision Tree Accuracy: {acc_tree:.4f}")
print(f"Random Forest Accuracy: {acc_forest:.4f}")


Decision Tree Accuracy: 0.7709
Random Forest Accuracy: 0.8324


**Analytical**  
Based on the accuracy results of the Decision Tree and Random Forest models, explain **why the Random Forest might perform better or worse** than a single Decision Tree.  
Discuss the roles of **ensemble learning**, **variance reduction**, and **overfitting** in your explanation.


**Answer:** Decision tree is intrinsicly very flexible but have high variance as small changes in the training data can lead to large changes in the structure and predictions, which can often results in overfitting where model memorizes the pattern rather than genralizing the unseen sample. On the other hand, in the random forest which is an ensemble model, it combines prediction reuslts frm multiple decision trees in which each tree is trained on bootstrap sample and uses random subset of features, which ensure uncorrelated errors after that predictions from these trees are aggregated using the majority vote and this average out the individual errors of the trees.This reuslts in better generalization and accuracy as in this case the accuracy has improved from 0.7989 to 0.8212 as the results of variance reductions and less sensitivity to noise data. 


---
## **Part 3 — Support Vector Machines (SVM) with Scikit-Learn**

In this final part, we explore another powerful supervised learning algorithm: the **Support Vector Machine (SVM)**. Unlike Decision Trees, which make splits by partitioning data, an SVM's goal is to find the optimal hyperplane or decision boundary that best separates the classes.

We will use `scikit-learn`'s implementation to train an SVM on the same preprocessed Titanic data and compare its performance to your from-scratch models.

### **Task: Train and Evaluate SVM Classifiers**

We will use `scikit-learn`'s `SVC` (Support Vector Classifier). We will experiment with two key hyperparameters: the `kernel` and the regularization parameter `C`.

#### **Understanding SVM Kernels**
The **kernel** allows the SVM to find complex, non-linear decision boundaries. We will use two types:
* **`linear` kernel:** This attempts to separate the data with a single straight line (or a flat plane in higher dimensions). It's fast and works well if the data is linearly separable. Think of it as using a ruler to divide two groups of points.
* **`rbf` kernel (Radial Basis Function):** This is the default and more flexible kernel. It can create complex, curved decision boundaries by mapping the data to a higher-dimensional space. Think of it as using a flexible wire to enclose different groups of points.

#### **Understanding the Regularization Parameter `C`**
The **`C` parameter** controls the trade-off between achieving a low training error and a low testing error.
* A **low `C`** makes the decision boundary smoother and the margin wider. The model is more tolerant of misclassifying a few training points, which can lead to better **generalization** (less overfitting).
* A **high `C`** aims to classify every training example correctly, which can lead to a more complex boundary and a narrower margin. This might **overfit** the training data.

**Instructions:**
1.  Use the same `X_train`, `y_train`, `X_test`, and `y_test` data from Part 1.
2.  Train one `SVC` model with a `linear` kernel as a baseline.
3.  Train three separate `SVC` models with an `rbf` kernel, using `C` values of `0.1`, `1`, and `10`.
4.  For each of the four models, calculate and print its accuracy on the test set.

In [ ]:


from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

#1. scale the features 
scalar = StandardScaler()
X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform (X_test)



#2. train and evaluate an SVM with a linear kernel
svm_linear = SVC (kernel ='linear', C=1)
svm_linear.fit(X_train_scaled, y_train)
y_pred_linear = svm_linear.predict(X_test_scaled)
acc_svm_linear = accuracy_score(y_test, y_pred_linear)


# 2. train and evaluate SVMs with an RBF kernel and different C values
rbf_accuracies = {}
for c_value in [0.1, 1, 10]:
    svm_rbf = SVC(kernel='rbf', C=c_value)
    svm_rbf.fit(X_train_scaled, y_train)
    y_pred_rbf = svm_rbf.predict(X_test_scaled)
    acc_rbf = accuracy_score(y_test, y_pred_rbf)
    rbf_accuracies[c_value] = acc_rbf
    
    
    
print(f"Custom Decision Tree Accuracy: {acc_tree:.4f}")
print(f"Custom Random Forest Accuracy: {acc_forest:.4f}")
print(f"Scikit-learn Linear SVM Accuracy: {acc_svm_linear:.4f}")
for c, acc in rbf_accuracies.items():
    print(f"Scikit-learn RBF SVM Accuracy (C={c}): {acc:.4f}")


Custom Decision Tree Accuracy: 0.7989
Custom Random Forest Accuracy: 0.8101
Scikit-learn Linear SVM Accuracy: 0.7821
Scikit-learn RBF SVM Accuracy (C=0.1): 0.8101
Scikit-learn RBF SVM Accuracy (C=1): 0.8156
Scikit-learn RBF SVM Accuracy (C=10): 0.7989


### **Final Comparison and Analysis**

Now, let's compare the performance of all the models you've worked with.

In [57]:
#Replace the accuracy variables with your defined variables for SVM accuracies

print(f"Custom Decision Tree Accuracy: {acc_tree:.4f}")
print(f"Custom Random Forest Accuracy: {acc_forest:.4f}")
print(f"Scikit-learn Linear SVM Accuracy: {acc_svm_linear:.4f}")
for c, acc in rbf_accuracies.items():
    print(f"Scikit-learn RBF SVM Accuracy (C={c}): {acc:.4f}")

Custom Decision Tree Accuracy: 0.7989
Custom Random Forest Accuracy: 0.8212
Scikit-learn Linear SVM Accuracy: 0.7821
Scikit-learn RBF SVM Accuracy (C=0.1): 0.8101
Scikit-learn RBF SVM Accuracy (C=1): 0.8156
Scikit-learn RBF SVM Accuracy (C=10): 0.7989


**Analysis**
1.  Looking at the RBF SVM results, how did changing the `C` parameter affect the model's accuracy? Based on the explanation above, what does your result suggest about the trade-off between a wide margin and fitting the training data for this problem?
2.  Compared to your Random Forest, what is one advantage and one disadvantage of using an SVM? (Consider factors like interpretability, training time, and prediction performance).

**Answer:**
1. When value of C increases this effect the model's accuracy, with increasing value of C the accuracy improves, at lower values of C model actually priotize widde margin over individual fitting data points, and it allows model tokeep the decision boundry smoother. While when the values of C are high the model minimizes the missclassifications, fitting the training data more closely, resulting in better accuracy due to narrow margins. But in case the value becomes very high ( as when C=10 )this can results in the overfitting of the model i.e. model will perform well on the teaing but poorly on the testing data and accuracy starts to drop down.  
2. SVM  requires careful preprocessing (feature scaling and tuning), which is relatively slower to train particularly when dataset is very large, though it outperform when features space is large and very complex realtive to number of samples as the kernel allows it to handle these complex boundries in such space. Moreover, it is very sensitive to the scaling and change in the hyperparameters can drastically change the reuslts. On the other hand,  Rnadom forest handles mixed features and non linearlity autmatically and is relatively less sensitive to scaling and even with less tuning provide better and higher accuracy. 